# PRISM Colab Runner

This notebook clones PRISM from GitHub, installs it in editable mode, runs smoke checks, generates a small ToyFireEnv dataset, trains a debug model, evaluates prediction metrics, and displays the generated figures.

Recommended Colab runtime: **Runtime > Change runtime type > T4 GPU**.

In [ ]:
# ===== User settings =====
REPO_URL = "https://github.com/KKKKKK-y/prism.git"
BRANCH = "main"

# If the repository is private, paste a GitHub token here temporarily.
# Leave empty for a public repository. Do not print this value.
GITHUB_TOKEN = ""

PROJECT_DIR = "/content/prism"

# Small, fast Colab run. Increase these when you want a stronger dataset.
SMALL_TRAIN_EPISODES = 5
SMALL_VAL_EPISODES = 2
SMALL_TEST_EPISODES = 2

# Formal run from README. This can take much longer.
RUN_FORMAL_PIPELINE = False
FORMAL_TRAIN_EPISODES = 300
FORMAL_VAL_EPISODES = 60
FORMAL_TEST_EPISODES = 60

In [ ]:
import getpass
import os
import shutil
import subprocess
from pathlib import Path

def run(cmd, cwd=None, check=True):
    safe_cmd = ["<TOKEN>" if GITHUB_TOKEN and part == GITHUB_TOKEN else part for part in cmd] if isinstance(cmd, list) else cmd
    print("\n$", " ".join(safe_cmd) if isinstance(safe_cmd, list) else safe_cmd)
    return subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str), check=check)

def tokenized_url(repo_url, token):
    return repo_url.replace("https://", f"https://{token}@")

if Path(PROJECT_DIR).exists():
    shutil.rmtree(PROJECT_DIR)

clone_result = run(["git", "clone", "--branch", BRANCH, REPO_URL, PROJECT_DIR], check=False)
if clone_result.returncode != 0:
    print("Public clone failed. If this is a private repo, paste a GitHub token with repo read access.")
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = getpass.getpass("GitHub token: ")
    run(["git", "clone", "--branch", BRANCH, tokenized_url(REPO_URL, GITHUB_TOKEN), PROJECT_DIR])

os.chdir(PROJECT_DIR)
print("Project:", Path.cwd())

In [ ]:
# Install PRISM. Colab usually already includes a CUDA-enabled PyTorch build.
run(["python", "-m", "pip", "install", "-U", "pip"])
run(["python", "-m", "pip", "install", "-e", "."])

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
# Quick project checks.
run(["python", "scripts/check_project_ready.py"])
run(["python", "scripts/test_env.py"])
run(["python", "scripts/test_shapes.py", "--config", "configs/smoke.yaml"])
run(["python", "scripts/test_planner.py", "--config", "configs/smoke.yaml"])

In [ ]:
# Generate a small ToyFireEnv dataset for a fast end-to-end Colab run.
run([
    "python", "scripts/generate_toy_dataset.py",
    "--config", "configs/toy_train.yaml",
    "--train_episodes", str(SMALL_TRAIN_EPISODES),
    "--val_episodes", str(SMALL_VAL_EPISODES),
    "--test_episodes", str(SMALL_TEST_EPISODES),
])
run(["python", "scripts/test_toy_dataset.py", "--npz", "outputs/datasets/toy_fire_train.npz"])

In [ ]:
# Debug training: one epoch with limited samples. This should finish quickly.
run(["python", "scripts/train.py", "--config", "configs/toy_train.yaml", "--debug"])

In [ ]:
# Evaluate predictions and create figures.
run([
    "python", "scripts/evaluate_prediction_on_toy.py",
    "--config", "configs/toy_train.yaml",
    "--checkpoint", "outputs/checkpoints_toy/best.pt",
])
run([
    "python", "scripts/plot_training_curve.py",
    "--csv", "outputs/results/stage4_toy_training_log.csv",
    "--output", "outputs/visualizations/stage4_toy_training_curve.png",
])
run([
    "python", "scripts/plot_prediction_metrics.py",
    "--csv", "outputs/results/stage4_toy_prediction_metrics.csv",
    "--output", "outputs/visualizations/stage4_toy_prediction_metrics.png",
])
run([
    "python", "scripts/visualize_prediction.py",
    "--config", "configs/toy_train.yaml",
    "--checkpoint", "outputs/checkpoints_toy/best.pt",
    "--output", "outputs/visualizations/stage4_toy_prediction_horizons.png",
    "--all_horizons",
])

In [ ]:
# Display generated outputs.
from IPython.display import Image, display

for image_path in [
    "outputs/visualizations/stage4_toy_training_curve.png",
    "outputs/visualizations/stage4_toy_prediction_metrics.png",
    "outputs/visualizations/stage4_toy_prediction_horizons.png",
]:
    path = Path(image_path)
    if path.exists():
        print(path)
        display(Image(filename=str(path)))
    else:
        print("Missing:", path)

In [ ]:
# Optional formal pipeline. Set RUN_FORMAL_PIPELINE = True in the first cell before running this.
if RUN_FORMAL_PIPELINE:
    run([
        "python", "scripts/generate_toy_dataset.py",
        "--config", "configs/toy_train.yaml",
        "--train_episodes", str(FORMAL_TRAIN_EPISODES),
        "--val_episodes", str(FORMAL_VAL_EPISODES),
        "--test_episodes", str(FORMAL_TEST_EPISODES),
    ])
    run(["python", "scripts/run_stage4_3_pipeline.py", "--config", "configs/toy_train.yaml"])
else:
    print("RUN_FORMAL_PIPELINE is False. Skipping formal training.")

## Save Outputs To Google Drive

Run the next cell if you want to keep checkpoints, datasets, CSV logs, and visualizations after the Colab session ends.

In [ ]:
# Optional: save outputs to Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# run(['bash', '-lc', 'mkdir -p /content/drive/MyDrive/prism_outputs && cp -r outputs/* /content/drive/MyDrive/prism_outputs/'])